In [ ]:
import spacy
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from collections import Counter
import numpy as np

/home/martavidas/anaconda3/envs/dipl_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#!python -m spacy download en_core_web_sm

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Load corpora from JSON files

In [3]:
import json
from pathlib import Path

def load_json_corpus(path, corpus_name):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            pairs.append((" ".join(entry["tokens"]), corpus_name))
    return pairs

# Example: adjust file paths to your actual files
NCBI_train = Path("/home/martavidas/Documents/FER/Diplomski/Diplomski/data/ncbi_disease_json/train.json")
generated_train = Path("/home/martavidas/Documents/FER/Diplomski/Diplomski/data/gen2_json/train.json")

pairs = load_json_corpus(NCBI_train, "NCBI_train") + load_json_corpus(generated_train, "generated_train")
sentences, corpus_labels = zip(*pairs)
print(f"Loaded {len(sentences)} sentences: "
      f"{corpus_labels.count('NCBI_train')} from NCBI_train and "
      f"{corpus_labels.count('generated_train')} from Generated.")


Loaded 14202 sentences: 5423 from NCBI_train and 8779 from Generated.


Extract pos and dp and save to a file

In [5]:
nlp = spacy.load("en_core_web_sm")

def extract_syntax_features(sentences, corpus_labels, output_path):
    features = []
    for i, (sent, corpus_name) in enumerate(zip(sentences, corpus_labels), start=1):
        doc = nlp(sent)
        pos_tags = [token.pos_ for token in doc]
        dep_rels = [token.dep_ for token in doc]
        features.append({
            "id": i,
            "pos": pos_tags,
            "dep": dep_rels,
            "corpus": corpus_name,
        })
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(features, f, indent=2)

extract_syntax_features(sentences, corpus_labels, "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/syntax_features.json")

Extract SRL and save to a file (run in nlp_srl environment)

nlp_srl environment creation:
conda create -n nlp_srl python=3.8 -y
conda activate nlp_srl
pip install torch==1.9.0 torchvision==0.10.0
pip install allennlp==2.10.1 allennlp-models==2.10.1
pip install spacy==3.2.4
pip install "transformers==4.4.2"


In [12]:
from allennlp.predictors.predictor import Predictor
import allennlp_models.tagging
import allennlp_models.coref

model_path = "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/structured-prediction-srl-bert.2020.12.15.tar.gz"
predictor = Predictor.from_path(model_path)

def extract_srl(sentences, output_path):
    srl_features = []
    id_counter = 1
    for sent in sentences:
        try:
            result = predictor.predict(sentence=sent)
        except Exception as e:
            print("SRL error:", e)
            return []
        patterns = []
        for verb_info in result.get("verbs", []):
            roles = verb_info["tags"]
            tokens = result["words"]
            frame = []
            for tok, role in zip(tokens, roles):
                if role != "O":
                    label = role.replace("B-", "").replace("I-", "")
                    frame.append(f"{label}:{tok.lower()}")
            if frame:
                patterns.append(" ".join(frame))
        srl_features.append({
            "id": id_counter,
            "srl": patterns
        })
        id_counter += 1
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(srl_features, f, indent=2)
extract_srl(sentences, "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/syntsrl_features.json")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Merge feature representations

In [7]:
def merge_features(syntax_file, srl_file, output_path):
    with open(syntax_file, "r", encoding="utf-8") as f:
        syntax = json.load(f)
    with open(srl_file, "r", encoding="utf-8") as f:
        srl = json.load(f)
    
    merged = []
    for s_syn, s_srl in zip(syntax, srl):
        assert s_syn.get("id") == s_srl.get("id")
        merged.append({
            "id": s_syn.get("id"),
            "pos": s_syn["pos"],
            "dep": s_syn["dep"],
            "srl": s_srl["srl"],
            "corpus": s_syn["corpus"],
        })
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2)

merge_features(
        "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/syntax_features.json",
        "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/syntsrl_features.json",
        "/home/martavidas/Documents/FER/Diplomski/paper/clusterTextSemantic/merged_features.json"
    )
